In [49]:
%pip install -qq pandas faker factory-boy 


Note: you may need to restart the kernel to use updated packages.


c:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\.venv\Scripts\python.exe: No module named pip


In [50]:
import pandas as pd
import factory

In [51]:
class PatientFactory(factory.Factory):
    class Meta:
        model = dict

    patient_id = factory.Sequence(lambda n: f"PT{n:04d}")
    age = factory.Faker('random_int', min=18, max=90)

In [52]:
class StudySiteFactory(factory.Factory):
    class Meta:
        model = dict

    site_id = factory.Sequence(lambda n: f"SITE{n:03d}")
    site_name = factory.Faker('company')
    

In [53]:
class StudyVisitFactory(factory.Factory):
    class Meta:
        model = dict

    visit_id = factory.Sequence(lambda n: f"VISIT{n:05d}")
    # Values are injected from StudyFactory to ensure concrete IDs, not declarations.
    patient_id = None
    site_id = None
    visit_date = factory.Faker('date_this_year')

In [54]:
import random

class StudyFactory(factory.Factory):
    class Meta:
        model = dict

    Name = factory.Faker('sentence', nb_words=3)
    StartDate = factory.Faker('date_this_decade')
    EndDate = factory.Faker('date_this_decade')
    Sites = factory.LazyFunction(lambda: [StudySiteFactory() for _ in range(3)])
    Participants = factory.LazyFunction(lambda: [PatientFactory() for _ in range(10)])
    Visits = factory.LazyAttribute(
        lambda obj: [
            StudyVisitFactory(
                patient_id=random.choice([p['patient_id'] for p in obj.Participants]),
                site_id=random.choice([s['site_id'] for s in obj.Sites]),
            )
            for _ in range(5)
        ]
    )

In [55]:
# Generate a random dataset
study = StudyFactory()
# Convert to DataFrame for display
study

{'Name': 'Beautiful.',
 'StartDate': datetime.date(2020, 7, 15),
 'EndDate': datetime.date(2024, 5, 9),
 'Sites': [{'site_id': 'SITE000', 'site_name': 'Gilbert and Sons'},
  {'site_id': 'SITE001', 'site_name': 'Flores-Swanson'},
  {'site_id': 'SITE002', 'site_name': 'Stanley Ltd'}],
 'Participants': [{'patient_id': 'PT0000', 'age': 33},
  {'patient_id': 'PT0001', 'age': 48},
  {'patient_id': 'PT0002', 'age': 32},
  {'patient_id': 'PT0003', 'age': 20},
  {'patient_id': 'PT0004', 'age': 36},
  {'patient_id': 'PT0005', 'age': 57},
  {'patient_id': 'PT0006', 'age': 55},
  {'patient_id': 'PT0007', 'age': 27},
  {'patient_id': 'PT0008', 'age': 20},
  {'patient_id': 'PT0009', 'age': 37}],
 'Visits': [{'visit_id': 'VISIT00000',
   'patient_id': 'PT0004',
   'site_id': 'SITE002',
   'visit_date': datetime.date(2026, 1, 20)},
  {'visit_id': 'VISIT00001',
   'patient_id': 'PT0004',
   'site_id': 'SITE002',
   'visit_date': datetime.date(2026, 3, 23)},
  {'visit_id': 'VISIT00002',
   'patient_id':

In [83]:
import sys
from pathlib import Path

def resolve_duckdb_file() -> str:
    # In JupyterLite/Pyodide, use a local file in the virtual FS.
    if sys.platform == "emscripten":
        return "clinical_study.db"

    # In desktop Python, keep using the repo's content/data location.
    db_path = (Path.cwd() / "../data/clinical_study.db").resolve()
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return str(db_path)

duckdb_file = resolve_duckdb_file()
print(f"DuckDB file: {duckdb_file}")

DuckDB file: C:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\content\data\clinical_study.db


In [84]:
# Delete existing database file if it exists
import os
if os.path.exists(duckdb_file):
    os.remove(duckdb_file)

In [85]:
# Write generated study to DuckDB database
import duckdb
# Create a connection to DuckDB
conn = duckdb.connect(duckdb_file)


In [86]:
# Create tables and insert data
conn.execute("""
CREATE TABLE IF NOT EXISTS Study (
    Name TEXT,
    StartDate DATE,
    EndDate DATE
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudySite (
    site_id TEXT,
    site_name TEXT
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS Patient (
    patient_id TEXT,
    age INTEGER
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudyVisit (
    visit_id TEXT,
    patient_id TEXT,
    site_id TEXT,
    visit_date DATE
)""")


In [87]:
# Insert study data
conn.execute("INSERT INTO Study (Name, StartDate, EndDate) VALUES (?, ?, ?)", (study['Name'], study['StartDate'], study['EndDate']))
# Insert site data
for site in study['Sites']:
    conn.execute("INSERT INTO StudySite (site_id, site_name) VALUES (?, ?)", (site['site_id'], site['site_name']))
# Insert patient data
for patient in study['Participants']:
    conn.execute("INSERT INTO Patient (patient_id, age) VALUES (?, ?)", (patient['patient_id'], patient['age']))
# Insert visit data
for visit in study['Visits']:
    conn.execute("INSERT INTO StudyVisit (visit_id, patient_id, site_id, visit_date) VALUES (?, ?, ?, ?)", 
                 (visit['visit_id'], visit['patient_id'], visit['site_id'], visit['visit_date']))
# Commit changes and keep the connection open for verification cells.
conn.commit()

In [88]:
# Read data back from DuckDB to verify
tables = conn.execute("SHOW TABLES").fetchdf()
print("Available tables:")
print(tables)

study_df = conn.execute("SELECT * FROM Study").fetchdf()
sites_df = conn.execute("SELECT * FROM StudySite").fetchdf()
patients_df = conn.execute("SELECT * FROM Patient").fetchdf()
visits_df = conn.execute("SELECT * FROM StudyVisit").fetchdf()

print("Study Data:")
print(study_df)
print("Sites Data:")
print(sites_df)
print("Patients Data:")
print(patients_df)
print("Visits Data:")
print(visits_df)

Available tables:
         name
0     Patient
1       Study
2   StudySite
3  StudyVisit
Study Data:
         Name  StartDate    EndDate
0  Beautiful. 2020-07-15 2024-05-09
Sites Data:
   site_id         site_name
0  SITE000  Gilbert and Sons
1  SITE001    Flores-Swanson
2  SITE002       Stanley Ltd
Patients Data:
  patient_id  age
0     PT0000   33
1     PT0001   48
2     PT0002   32
3     PT0003   20
4     PT0004   36
5     PT0005   57
6     PT0006   55
7     PT0007   27
8     PT0008   20
9     PT0009   37
Visits Data:
     visit_id patient_id  site_id visit_date
0  VISIT00000     PT0004  SITE002 2026-01-20
1  VISIT00001     PT0004  SITE002 2026-03-23
2  VISIT00002     PT0003  SITE000 2026-03-12
3  VISIT00003     PT0008  SITE000 2026-04-03
4  VISIT00004     PT0002  SITE002 2026-01-03


In [89]:
# Run a simple query to join tables and verify relationships
query = """
SELECT v.visit_id, v.visit_date, p.patient_id, p.age, s.site_id, s.site_name
FROM StudyVisit v
JOIN Patient p ON v.patient_id = p.patient_id
JOIN StudySite s ON v.site_id = s.site_id
"""
joined_df = conn.execute(query).fetchdf()
print(joined_df)

# Close connection when all reads are complete.
conn.close()

     visit_id visit_date patient_id  age  site_id         site_name
0  VISIT00004 2026-01-03     PT0002   32  SITE002       Stanley Ltd
1  VISIT00002 2026-03-12     PT0003   20  SITE000  Gilbert and Sons
2  VISIT00001 2026-03-23     PT0004   36  SITE002       Stanley Ltd
3  VISIT00003 2026-04-03     PT0008   20  SITE000  Gilbert and Sons
4  VISIT00000 2026-01-20     PT0004   36  SITE002       Stanley Ltd
